[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/fractals/fractals.ipynb)

# Fractals of the Even Subalgebra

This example renders the Mandelbrot set in the plane and rotating planar slices of a four-dimensional Julia set in space on the GPU. Both sets arise from iterating a square plus a constant: the Mandelbrot set records which constants keep the orbit starting at the constant itself bounded, while a Julia set records which starting values stay bounded for one fixed constant.

Each iterated value is modelled as an even multivector combining a turn and a scale. In the plane an even multivector is a scalar plus a multiple of `xy`, and in three-dimensional space it is a scalar plus a bivector turning in that bivector's plane; in both settings the geometric square `z * z` doubles the turn angle in the same plane and squares the magnitude, so `z * z + constant` runs unchanged as a single `numga` WGSL shader compiled against either algebra while a rotor `turn >> constant_plane` sweeps the viewing plane through the four-dimensional Julia set.

In complex-number and quaternion notation, the planar iteration is written $z_{n+1} = z_n^2 + c$ with $z, c \in \mathbb C$ and $i^2 = -1$, whereas the four-dimensional iteration introduces three separate imaginary units $q = w + x\,\mathbf i + y\,\mathbf j + z\,\mathbf k$ governed by $\mathbf i^2 = \mathbf j^2 = \mathbf k^2 = \mathbf i\mathbf j\mathbf k = -1$ and rotates the slicing basis with an explicit $4 \times 4$ orthogonal matrix.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext
from numga.algebras import VGA2D, VGA3D
from examples.util.animation import save_figure, show_animation, show_states
from examples.math.fractals import render
from numga.wgsl import Shader, supersampled
from examples.util.panels import image_figure

plane_algebra = VGA2D
space_algebra = VGA3D
plane_context = NumpyContext(plane_algebra)
space_context = NumpyContext(space_algebra)
plane = plane_context.multivector
space = space_context.multivector
Bivector = space_algebra.gatype.bivector()                                      # Bivector+

## Even multivectors of the plane

An even multivector of the plane combines a scalar with a multiple of the unit bivector `plane.xy`. Multiplying a vector on the right by `plane.xy` turns it by a quarter turn (`plane.x * plane.xy == plane.y`), so squaring `plane.xy` applies a half turn and evaluates to minus one.

In [ ]:
print(plane.xy.squared().coefficient_table())

## Escape-time iteration in a shader

Any orbit whose norm exceeds two escapes to infinity, so iterating past a much larger radius leaves the set unchanged while letting the final norm interpolate fractional escape counts smoothly between integer steps. Because `z * z + constant` uses only the even geometric product and `z.scalar_norm_squared()`, one shader written in the `numga` WGSL dialect compiles directly for any algebra's even subalgebra, with `supersampled()` averaging a subpixel grid in every pixel.

In [ ]:
escape_source = render.colouring() + '''
alias Sample = vec3<f32>;
var<uniform> n_steps: u32;

// Squared bailout radius large enough for smooth fractional escape-count interpolation.
const bailout_squared = 256.0;

fn escape(start: Even, constant: Even) -> vec3<f32> {
    var z = start;
    for (var step = 0u; step < n_steps; step++) {
        let size = z.scalar_norm_squared();
        if (size > bailout_squared) {
            return banded(f32(step) + 1 - log2(log2(size) / 2));
        }
        z = z * z + constant;
    }
    return bounded;
}
'''


def escape_shader(sample_source: str, algebra: Algebra) -> Shader:
    """Compiles `sample_source` together with the supersampled escape loop against the even and bivector types of `algebra`."""
    return Shader(supersampled() + escape_source + sample_source, algebra, dict(Even=algebra.gatype.even(), Bivector=algebra.gatype.bivector()), {})

## Mandelbrot set

In the Mandelbrot set each screen sample supplies both the starting value and the additive constant, mapping horizontal screen offset to the scalar part and vertical offset to `plane.xy`.

In [ ]:
mandelbrot_centre = plane.scalar([-0.7])                                        # [] Scalar+
mandelbrot_half_height = 1.15
n_mandelbrot_steps = 200
mandelbrot_shape = (220, 300)

In [ ]:
mandelbrot = escape_shader('''
var<uniform> centre: Even;
var<uniform> half_height: f32;

fn sample(place: vec2<f32>) -> Sample {
    let offset = screen(place);
    let point = centre + (offset.x + mv.xy * offset.y) * half_height;
    return escape(point, point);
}
''', plane_algebra)

overview = mandelbrot(mandelbrot_shape, centre=mandelbrot_centre, half_height=mandelbrot_half_height,
                      n_steps=n_mandelbrot_steps)["image"]
save_figure(image_figure(render.picture(overview)), "mandelbrot");

Dark pixels belong to constants whose orbits remain bounded, surrounded by repeating colour bands that cycle faster as escape times climb toward the boundary.

## Zooming into Seahorse Valley

Shrinking the view height by a constant geometric ratio each frame zooms into the valley between the main cardioid and the period-two disc, where self-similar spirals line the boundary at every scale and finer views raise the iteration limit accordingly.

In [ ]:
zoom_centre = -0.743644 + 0.131826 * plane.xy                                   # [] Even+
zoom_half_heights = (1.2, 0.002)
n_zoom_step_bounds = (150, 600)
n_zoom_frames = 126
zoom_duration_ms = 40
zoom_shape = (225, 300)

In [ ]:
half_heights = np.geomspace(*zoom_half_heights, n_zoom_frames)                  # [n_zoom_frames]
step_counts = np.linspace(*n_zoom_step_bounds, n_zoom_frames).round().astype(int)  # [n_zoom_frames]
zoom_frames = (render.picture(mandelbrot(zoom_shape, centre=zoom_centre, half_height=half_height, n_steps=n_steps)["image"])
               for half_height, n_steps in zip(half_heights, step_counts))
show_animation(zoom_frames, "mandelbrot_zoom", zoom_duration_ms);

## Rotating slices of a four-dimensional Julia set

In three-dimensional space, an even multivector has one scalar and three bivector components, making its Julia set four-dimensional. Sampling starts of the form `offset.x + up * offset.y` cuts a two-dimensional plane spanned by scalars and a unit bivector `up`: when `up` aligns with the normalized bivector part of `julia_constant`, the starts and the constant share a single rotation plane and reproduce a planar Julia set, and turning `up` toward `space.xy` by a rotor sweeps through oblique slices of the four-dimensional boundary.

In [ ]:
julia_constant = -0.2 + 0.6 * space.yz + 0.2 * space.zx                         # [] Even+
julia_half_height = 1.5
julia_shape = (240, 240)
n_julia_steps = 40
n_frames = 90
duration_ms = 40

In [ ]:
julia_source = '''
var<uniform> constant: Even;
var<uniform> up: Bivector;
var<uniform> half_height: f32;

fn sample(place: vec2<f32>) -> Sample {
    let offset = screen(place);
    return escape((offset.x + up * offset.y) * half_height, constant);
}
'''
julia = escape_shader(julia_source, space_algebra)


def slice_frames(n_frames: int) -> Iterator[tuple[Bivector, np.ndarray]]:
    """Yields each frame's slicing unit bivector (`[] Bivector+`) and float RGB slice image (`[rows, columns, 3]`) as a rotor turns the constant's plane toward `xy`."""
    constant_plane = julia_constant.select_grade(2).normalized()                # [] Bivector+
    # Commutator bivector spanning the rotation from `constant_plane` toward `space.xy`.
    tilt = space.xy.commutator(constant_plane).normalized()                     # [] Bivector+
    for angle in np.linspace(0.0, np.pi / 2, n_frames):
        turn = (tilt * (-angle / 2)).exp()                                      # [] Rotor+
        up = turn >> constant_plane                                             # [] Bivector+
        image = julia(julia_shape, constant=julia_constant, up=up, half_height=julia_half_height,
                      n_steps=n_julia_steps)["image"]
        yield up, image


julia_states = show_states(
    slice_frames(n_frames),
    lambda states: (render.picture(image) for _, image in states),
    name="julia_slices",
    duration_ms=duration_ms,
    n_frames=n_frames,
);

In [ ]:
# checks
# `xy` squares to -1, the Mandelbrot overview keeps the origin dark and escapes near the outer corners, and the initial
# 4D Julia slice in the constant's own plane matches the 2D Julia set with the same scalar and bivector norm.
assert (plane.xy.squared() + 1).abs() < 1e-12
centre_colour = overview[mandelbrot_shape[0] // 2, mandelbrot_shape[1] * 7 // 10]
corner_colour = overview[0, -1]
assert np.allclose(centre_colour, [0.02, 0.02, 0.04], atol=1e-3)
assert np.abs(corner_colour - centre_colour).max() > 0.1

first_up, first_slice = julia_states[0]
last_up, _ = julia_states[-1]
assert (first_up.norm() - 1).abs() < 1e-12
assert (last_up.norm() - 1).abs() < 1e-12
plane_constant = (plane_context.include_from_algebra(julia_constant.select_grade(0))
                  + plane_context.include_from_algebra(julia_constant.select_grade(2).norm()) * plane.xy)
plane_julia = escape_shader(julia_source, plane_algebra)(
    julia_shape, constant=plane_constant, up=plane.xy, half_height=julia_half_height, n_steps=n_julia_steps
)["image"]
assert (np.abs(first_slice - plane_julia).max(axis=-1) > 2 / 255).mean() < 0.01